# Fungsi TF-IDF

## 1. module dan library yang dibutuhkan

In [3]:
# Library Sys
import sys
sys.path.append('..')

# Library dan Module yang dibutuhkan
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from src import load_objek,simpan_objek

## 2. Load dataset

In [5]:
data = load_objek('../data/processed/data_teks_bersih.pkl')  
# Memuat data hasil preprocessing sebelumnya
# Berisi: X_train_clean, X_test_clean, X_test_noisy, y_train, y_test

# =========================
# LOAD DATA SPLIT
# =========================

train_df = pd.read_csv("../data/processed/data_train.csv")
test_df = pd.read_csv("../data/processed/data_test.csv")

X_train_raw = train_df["tweet"]
y_train = train_df["label"]

X_test_raw = test_df["tweet"]
y_test = test_df["label"]

## 2. Logika Code

### 2.1 Load teks yang sudah bersih

In [7]:
# 2. Ekstrasi Fitur (TF-IDF)
tfidf = TfidfVectorizer(
    ngram_range=(1,2),   # 🔥 penting (kata + frasa)
    max_features=5000,   # batasi fitur biar fokus
    min_df=2,            # buang kata jarang
    max_df=0.9           # buang kata terlalu umum
)

### 2.2 Melakukan tfidf pada dataset

In [8]:
# ============================================================
# TF-IDF TRAINING (SUDAH PUNYA KAMU)
# ============================================================

X_train_vec = tfidf.fit_transform(data['X_train_clean'])

X_test_clean_vec = tfidf.transform(data['X_test_clean'])
X_test_noisy_vec = tfidf.transform(data['X_test_noisy'])


# ============================================================
# DEBUG: CEK APAKAH TF-IDF ADA ISINYA
# ============================================================

print("Jumlah fitur:", len(tfidf.get_feature_names_out()))
print("Shape TF-IDF:", X_train_vec.shape)

# ============================================================
# AMBIL DATA REAL TANPA FILTER SALAH
# ============================================================

import pandas as pd

feature_names = tfidf.get_feature_names_out()

# Ambil 5 dokumen pertama
df_tfidf = pd.DataFrame(
    X_train_vec[:5].toarray(),
    columns=feature_names
)

# ============================================================
# 🔥 FILTER YANG NILAINYA TIDAK NOL
# ============================================================

df_non_zero = df_tfidf.loc[:, (df_tfidf != 0).any(axis=0)]

# Ambil maksimal 5 kolom saja biar rapi
df_non_zero = df_non_zero.iloc[:, :5]

# Tambahkan label dokumen
df_non_zero["Dokumen"] = [f"Berita {i+1}" for i in range(len(df_non_zero))]
df_non_zero.set_index("Dokumen", inplace=True)

print("\n=== HASIL TF-IDF (TANPA NILAI 0) ===")
print(df_non_zero)

Jumlah fitur: 5000
Shape TF-IDF: (3049, 5000)

=== HASIL TF-IDF (TANPA NILAI 0) ===
           agustus     bahan     bahas      buat  buat china
Dokumen                                                     
Berita 1  0.000000  0.000000  0.000000  0.261429     0.54172
Berita 2  0.295293  0.000000  0.000000  0.000000     0.00000
Berita 3  0.000000  0.000000  0.000000  0.000000     0.00000
Berita 4  0.000000  0.000000  0.239331  0.000000     0.00000
Berita 5  0.000000  0.158167  0.000000  0.113148     0.00000


#### 2.2.1 Mengambil hasil dari splitting dan TF-IDF

In [9]:
import pandas as pd

# ============================================================
# HITUNG JUMLAH DATA
# ============================================================

jumlah_train = len(X_train_raw)
jumlah_test = len(X_test_raw)
jumlah_total = jumlah_train + jumlah_test

# ============================================================
# HITUNG JUMLAH FITUR TF-IDF
# ============================================================

jumlah_fitur = X_train_vec.shape[1]  # jumlah kolom TF-IDF

# ============================================================
# HITUNG PERSENTASE
# ============================================================

persen_train = (jumlah_train / jumlah_total) * 100
persen_test = (jumlah_test / jumlah_total) * 100

# ============================================================
# BUAT DATAFRAME (TABEL)
# ============================================================

tabel_distribusi = pd.DataFrame({
    "Kategori Data": [
        "Data Latih (Training Set)",
        "Data Uji (Testing Set)",
        "Total Keseluruhan"
    ],
    "Persentase": [
        f"{persen_train:.0f}%",
        f"{persen_test:.0f}%",
        "100%"
    ],
    "Jumlah Sampel (Baris)": [
        jumlah_train,
        jumlah_test,
        jumlah_total
    ],
    "Jumlah Fitur (Kolom TF-IDF)": [
        jumlah_fitur,
        jumlah_fitur,
        jumlah_fitur
    ]
})

# ============================================================
# OUTPUT
# ============================================================

print("\n=== TABEL DISTRIBUSI DATA ===")
print(tabel_distribusi)


=== TABEL DISTRIBUSI DATA ===
               Kategori Data Persentase  Jumlah Sampel (Baris)  \
0  Data Latih (Training Set)        80%                   3049   
1     Data Uji (Testing Set)        20%                    763   
2          Total Keseluruhan       100%                   3812   

   Jumlah Fitur (Kolom TF-IDF)  
0                         5000  
1                         5000  
2                         5000  


### 2.3 Menyimpan hasil preprocesing 

In [35]:
data_vektor = {
    'X_train': X_train_vec,              # Fitur training dalam bentuk vektor
    'y_train': data['y_train'],          # Label training

    'X_test_clean': X_test_clean_vec,    # Fitur test bersih
    'y_test': data['y_test'],            # Label test

    'X_test_noisy': X_test_noisy_vec,    # Fitur test kotor (robustness test)

    'vectorizer': tfidf                 # Simpan model TF-IDF (penting untuk inference!)
}

# Menyimpan semua hasil ke file pickle
simpan_objek(data_vektor, '../data/processed/data_vektor.pkl')

print("Proses Feature Engineering (TF-IDF) Selesai!")

Proses Feature Engineering (TF-IDF) Selesai!
